# Chapter 8 - Data Modification: Propositions & Queries
**Name:** Amrina Qayyum  
**Group Number:** EOS_grp_1  
**Date:** April 12, 2026  
**Database:** Northwinds2024Student  

---



## Proposition 1
**Problem Statement:** Create a working table for customer records and insert a few sample rows manually using `INSERT ... VALUES`.

**Why This Query Is Special:** It demonstrates the most basic form of data modification and sets up a reusable demo table for later propositions.


In [ ]:
-- Proposition 1 Query
-- Database: Northwinds2024Student
USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.CustomerNotesDemo;
GO

CREATE TABLE dbo.CustomerNotesDemo
(
    NoteID INT IDENTITY(1,1) PRIMARY KEY,
    CustomerCode VARCHAR(10) NOT NULL,
    NoteText VARCHAR(200) NOT NULL,
    PriorityLevel INT NOT NULL,
    CreatedAt DATETIME2 NOT NULL DEFAULT SYSDATETIME()
);
GO

INSERT INTO dbo.CustomerNotesDemo (CustomerCode, NoteText, PriorityLevel)
VALUES
    ('ALFKI', 'Priority customer follow-up required', 1),
    ('ANATR', 'Requested product catalog update', 2),
    ('AROUT', 'Monitor recent order pattern', 3);
GO

SELECT *
FROM dbo.CustomerNotesDemo;


---
## Proposition 2
**Problem Statement:** Copy all customers from London into a demo table using `INSERT ... SELECT` so they can be analyzed separately.

**Why This Query Is Special:** It shows how to populate a table from an existing dataset using a filtered query, which is common in staging and reporting work.


In [ ]:
-- Proposition 2 Query
USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.LondonCustomersDemo;
GO

CREATE TABLE dbo.LondonCustomersDemo
(
    custid INT PRIMARY KEY,
    companyname NVARCHAR(100),
    contactname NVARCHAR(100),
    city NVARCHAR(100),
    country NVARCHAR(100)
);
GO

INSERT INTO dbo.LondonCustomersDemo (custid, companyname, contactname, city, country)
SELECT c.custid, c.companyname, c.contactname, c.city, c.country
FROM Sales.Customer AS c
WHERE c.city = N'London';
GO

SELECT *
FROM dbo.LondonCustomersDemo
ORDER BY companyname;


---
## Proposition 3
**Problem Statement:** Update a priority label for customers in the London demo table based on the number of orders they placed.

**Why This Query Is Special:** It uses `UPDATE` with a joined aggregate subquery, making it more meaningful than a simple fixed-value update.


In [ ]:
-- Proposition 3 Query
USE Northwinds2024Student;
GO

IF COL_LENGTH('dbo.LondonCustomersDemo', 'priority_status') IS NULL
BEGIN
    ALTER TABLE dbo.LondonCustomersDemo
    ADD priority_status VARCHAR(20) NULL;
END;
GO

WITH OrderCounts AS
(
    SELECT o.custid, COUNT(*) AS total_orders
    FROM Sales.[Order] AS o
    GROUP BY o.custid
)
UPDATE lcd
SET priority_status = CASE
                        WHEN oc.total_orders >= 10 THEN 'High'
                        WHEN oc.total_orders >= 5 THEN 'Medium'
                        ELSE 'Low'
                      END
FROM dbo.LondonCustomersDemo AS lcd
INNER JOIN OrderCounts AS oc
    ON lcd.custid = oc.custid;
GO

SELECT *
FROM dbo.LondonCustomersDemo
ORDER BY priority_status, companyname;


---
## Proposition 4
**Problem Statement:** Delete customer notes that are low priority after the team decides only urgent notes should remain.

**Why This Query Is Special:** It demonstrates controlled row removal with a meaningful business rule instead of deleting everything blindly.


In [ ]:
-- Proposition 4 Query
USE Northwinds2024Student;
GO

DELETE FROM dbo.CustomerNotesDemo
WHERE PriorityLevel = 3;
GO

SELECT *
FROM dbo.CustomerNotesDemo;


---
## Proposition 5
**Problem Statement:** Create a new snapshot table of discontinued products using `SELECT INTO`.

**Why This Query Is Special:** It is useful because `SELECT INTO` both creates the table and loads the data in one statement.


In [ ]:
-- Proposition 5 Query
USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.DiscontinuedProductsSnapshot;
GO

SELECT p.productid,
       p.productname,
       p.unitprice,
       p.discontinued
INTO dbo.DiscontinuedProductsSnapshot
FROM Production.Product AS p
WHERE p.discontinued = 1;
GO

SELECT *
FROM dbo.DiscontinuedProductsSnapshot
ORDER BY productname;


---
## Proposition 6
**Problem Statement:** Empty the discontinued product snapshot quickly after testing by using `TRUNCATE TABLE`.

**Why This Query Is Special:** It shows the difference between deleting rows and resetting a staging table efficiently.


In [ ]:
-- Proposition 6 Query
USE Northwinds2024Student;
GO

TRUNCATE TABLE dbo.DiscontinuedProductsSnapshot;
GO

SELECT COUNT(*) AS remaining_rows
FROM dbo.DiscontinuedProductsSnapshot;


---
## Proposition 7
**Problem Statement:** Use a CTE to find customers with many orders and update a loyalty flag in a reporting table.

**Why This Query Is Special:** It combines `CTE` logic with `UPDATE`, which is a stronger Chapter 8 example than using a CTE only for reading.


In [ ]:
-- Proposition 7 Query
USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.CustomerLoyaltyDemo;
GO

SELECT c.custid,
       c.companyname,
       CAST('Regular' AS VARCHAR(20)) AS loyalty_status
INTO dbo.CustomerLoyaltyDemo
FROM Sales.Customer AS c;
GO

WITH FrequentCustomers AS
(
    SELECT o.custid
    FROM Sales.[Order] AS o
    GROUP BY o.custid
    HAVING COUNT(*) >= 10
)
UPDATE cld
SET loyalty_status = 'Loyal'
FROM dbo.CustomerLoyaltyDemo AS cld
INNER JOIN FrequentCustomers AS fc
    ON cld.custid = fc.custid;
GO

SELECT TOP (20) *
FROM dbo.CustomerLoyaltyDemo
ORDER BY loyalty_status DESC, companyname;


---
## Proposition 8
**Problem Statement:** Keep only the five newest internal notes and remove older ones using `DELETE` with `OFFSET-FETCH` logic.

**Why This Query Is Special:** It applies pagination-style logic to data cleanup, which makes the query more advanced and practical.


In [ ]:
-- Proposition 8 Query
USE Northwinds2024Student;
GO

INSERT INTO dbo.CustomerNotesDemo (CustomerCode, NoteText, PriorityLevel)
VALUES
    ('BERGS', 'New shipment note', 2),
    ('BLAUS', 'Payment follow-up note', 2),
    ('BLONP', 'Discount review note', 1),
    ('BOLID', 'Retention note', 2),
    ('BONAP', 'Urgent service note', 1),
    ('BOTTM', 'Return management note', 2);
GO

DELETE FROM dbo.CustomerNotesDemo
WHERE NoteID IN
(
    SELECT NoteID
    FROM dbo.CustomerNotesDemo
    ORDER BY CreatedAt DESC, NoteID DESC
    OFFSET 5 ROWS
);
GO

SELECT *
FROM dbo.CustomerNotesDemo
ORDER BY CreatedAt DESC, NoteID DESC;


---
## Proposition 9
**Problem Statement:** Capture the before-and-after values of a price increase by using the `OUTPUT` clause during an update.

**Why This Query Is Special:** It is special because it records the exact changes made, which is very useful for auditing and verification.


In [ ]:
-- Proposition 9 Query
USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.PriceAuditDemo;
GO

CREATE TABLE dbo.PriceAuditDemo
(
    productid INT,
    productname NVARCHAR(100),
    oldprice MONEY,
    newprice MONEY,
    changedat DATETIME2 NOT NULL DEFAULT SYSDATETIME()
);
GO

UPDATE p
SET unitprice = unitprice * 1.05
OUTPUT inserted.productid,
       inserted.productname,
       deleted.unitprice,
       inserted.unitprice,
       SYSDATETIME()
INTO dbo.PriceAuditDemo(productid, productname, oldprice, newprice, changedat)
FROM Production.Product AS p
WHERE p.discontinued = 0
  AND p.unitprice < 20;
GO

SELECT TOP (20) *
FROM dbo.PriceAuditDemo
ORDER BY changedat DESC, productid;


---
## Proposition 10
**Problem Statement:** Synchronize a customer order summary table with the latest order totals using `MERGE`.

**Why This Query Is Special:** It is one of the strongest Chapter 8 statements because it handles insert and update logic together in one operation.


In [ ]:
-- Proposition 10 Query
USE Northwinds2024Student;
GO

DROP TABLE IF EXISTS dbo.CustomerOrderSummaryDemo;
GO

CREATE TABLE dbo.CustomerOrderSummaryDemo
(
    custid INT PRIMARY KEY,
    total_orders INT NOT NULL,
    last_order_date DATE NULL
);
GO

MERGE dbo.CustomerOrderSummaryDemo AS tgt
USING
(
    SELECT o.custid,
           COUNT(*) AS total_orders,
           MAX(o.orderdate) AS last_order_date
    FROM Sales.[Order] AS o
    GROUP BY o.custid
) AS src
ON tgt.custid = src.custid
WHEN MATCHED THEN
    UPDATE SET tgt.total_orders = src.total_orders,
               tgt.last_order_date = src.last_order_date
WHEN NOT MATCHED THEN
    INSERT (custid, total_orders, last_order_date)
    VALUES (src.custid, src.total_orders, src.last_order_date);
GO

SELECT TOP (20) *
FROM dbo.CustomerOrderSummaryDemo
ORDER BY total_orders DESC, last_order_date DESC;


NACE Competencies Reflection
Equity & Inclusion: In this assignment, I tried to write SQL statements in a clear and understandable way so that all group members could follow the logic and learn from the work equally.

Career & Self-Development: I improved my database and SQL skills by practicing data modification queries and learning how to organize solutions in a professional notebook format.

Communication: I presented each proposition and its matching SQL query in a structured way so the instructor and group members can easily understand the purpose of each task.

Critical Thinking: I analyzed the assignment requirements carefully and selected appropriate SQL operations such as insert, update, delete, and merge-like data handling to solve the problems 
correctly.
Professionalism: I followed the submission format, kept the work organized, and prepared the notebook in a neat and complete way for academic submission.

Technology: I used SQL and Jupyter Notebook effectively to create, document, and present database solutions in the required .ipynb format.

Leadership: I took initiative in preparing the notebook content and organizing the propositions and code so it can be handed over easily to the group leader.

Teamwork: Even though I worked on the technical part individually, the final notebook was prepared in a way that supports the group’s overall submission and can be shared with other members.